# Tests: `fasterai.core.ratio` (source `nbs/core/ratio.ipynb`)

In [ ]:
from fastcore.test import *
from fasterai.core.ratio import *

In [ ]:
from fastcore.test import *
import warnings

# Fractions pass through untouched (1 is 100%, not 1%)
test_eq(as_fraction(0.4, 'sparsity'), 0.4)
test_eq(as_fraction(0, 'sparsity'), 0.0)
test_eq(as_fraction(1, 'sparsity'), 1.0)
test_eq(as_fraction(0.999, 'sparsity'), 0.999)
assert isinstance(as_fraction(1, 'sparsity'), float)

# (1, 100] is read as a percent for one release, and says so
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    test_eq(as_fraction(40, 'sparsity'), 0.4)
    test_eq(as_fraction(100, 'sparsity'), 1.0)
    test_eq(as_fraction(1.5, 'sparsity'), 0.015)
test_eq(len(w), 3)
test_eq({x.category for x in w}, {FutureWarning})
_msg = str(w[0].message)
assert 'looks like a percent' in _msg, _msg
assert 'this argument is a fraction in [0, 1] (0.4 = 40%)' in _msg, _msg
assert 'pass the fraction' in _msg, _msg

# A fraction never warns
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    for v in (0, 0.25, 0.5, 1): as_fraction(v, 'sparsity')
test_eq(len(w), 0)

# Out of range raises, naming the argument and the layer
with ExceptionExpected(ValueError, regex='fraction'): as_fraction(150, 'sparsity')
with ExceptionExpected(ValueError, regex='fraction'): as_fraction(-0.1, 'sparsity')
with ExceptionExpected(ValueError, regex="layer1.0.conv1"):
    as_fraction(150, 'pruning_ratio', layer='layer1.0.conv1')

# Non-numbers raise TypeError — booleans are not ratios
with ExceptionExpected(TypeError, regex='must be a number'): as_fraction('0.4', 'sparsity')
with ExceptionExpected(TypeError, regex='must be a number'): as_fraction(True, 'sparsity')
with ExceptionExpected(TypeError, regex='must be a number'): as_fraction(None, 'sparsity')
with ExceptionExpected(TypeError, regex='must be a number'): as_fraction([0.4], 'sparsity')

# 0 is "leave this one alone", except where nothing would be removed
test_eq(as_fraction(0, 'pruning_ratio', layer='fc'), 0.0)
with ExceptionExpected(ValueError, regex='removes nothing'):
    as_fraction(0, 'pruning_ratio', allow_zero=False)

# Idempotent: converting twice neither changes the value nor warns again
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _once = as_fraction(40, 'sparsity')
    _twice = as_fraction(_once, 'sparsity')
test_eq(_once, _twice)
test_eq(len(w), 1)

# numpy scalars are numbers too (to_layer_targets returns them)
import numpy as np
test_eq(as_fraction(np.float64(0.4), 'sparsity'), 0.4)